<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


<h1>实验：使用动量训练神经网络</h1>


<h2>本笔记本目标</h2>

<h5> 1. 训练具有不同动量参数值的神经网络模型。</h5>
<h5> 2. 比较不同动量项的结果。 </h5>     



<h2>目录</h2>
<p>在本实验中，你将了解动量参数的不同取值如何影响神经网络的收敛速度。</p>

- [神经网络模块与训练函数](#Neural-Network-Module-and-Function-for-Training)
- [训练具有不同动量参数值的神经网络](#Train-Different-Networks-Model-different-values-for-the-Momentum-Parameter)
- [比较不同动量项的结果](#Compare-Results-of-Different-Momentum-Terms)

<p>预计所需时间：<strong>25 分钟</strong></p>

<hr>


<h2>准备工作</h2>


我们需要以下库：


In [ ]:
%%time
%pip install numpy matplotlib
%pip install torch==2.8.0+cpu torchvision==0.23.0+cpu torchaudio==2.8.0+cpu \
    --index-url https://download.pytorch.org/whl/cpu

In [ ]:
# 导入本实验所需的库

# 用于绘制数据和损失曲线
import matplotlib.pyplot as plt 
# 允许我们使用数组来操作和存储数据
import numpy as np
# PyTorch 库
import torch
# PyTorch 神经网络
import torch.nn as nn
# 允许我们使用激活函数
import torch.nn.functional as F
# 用于绘制数据和损失曲线
from matplotlib.colors import ListedColormap
# 用于帮助创建数据集并执行小批量训练
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(1)
np.random.seed(1)

用于绘图的函数：


In [ ]:
# 定义绘制决策区域的函数

def plot_decision_regions_3class(model, data_set):
    cmap_light = ListedColormap(['#FFAAAA', '#AAFFAA','#00AAFF'])
    cmap_bold = ListedColormap(['#FF0000', '#00FF00','#00AAFF'])
    X=data_set.x.numpy()
    y=data_set.y.numpy()
    h = .02
    x_min, x_max = X[:, 0].min() - 0.1 , X[:, 0].max() + 0.1 
    y_min, y_max = X[:, 1].min() - 0.1 , X[:, 1].max() + 0.1 
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),np.arange(y_min, y_max, h))
    XX=torch.torch.Tensor(np.c_[xx.ravel(), yy.ravel()])
    _,yhat=torch.max(model(XX),1)
    yhat=yhat.numpy().reshape(xx.shape)
    plt.pcolormesh(xx, yy, yhat, cmap=cmap_light, shading='auto')
    plt.plot(X[y[:]==0,0], X[y[:]==0,1], 'ro', label='y=0')
    plt.plot(X[y[:]==1,0], X[y[:]==1,1], 'go', label='y=1')
    plt.plot(X[y[:]==2,0], X[y[:]==2,1], 'o', label='y=2')
    plt.title("decision region")
    plt.legend()

创建数据集类：我们将在下面展示数据集


In [ ]:
# 创建数据集类

class Data(Dataset):
    
    # 修改自：http://cs231n.github.io/neural-networks-case-study/
    # 构造函数
    def __init__(self, K=3, N=500):
        D = 2
        X = np.zeros((N * K, D)) # data matrix (each row = single example)
        y = np.zeros(N * K, dtype='uint8') # class labels
        for j in range(K):
          ix = range(N * j, N * (j + 1))
          r = np.linspace(0.0, 1, N) # radius
          t = np.linspace(j * 4, (j + 1) * 4, N) + np.random.randn(N) * 0.2 # theta
          X[ix] = np.c_[r * np.sin(t), r * np.cos(t)]
          y[ix] = j
    
        self.y = torch.from_numpy(y).type(torch.LongTensor)
        self.x = torch.from_numpy(X).type(torch.FloatTensor)
        self.len = y.shape[0]
            
    # 获取方法
    def __getitem__(self, index):    
        return self.x[index], self.y[index]
    
    # 获取长度
    def __len__(self):
        return self.len
    
    # 绘制示意图
    def plot_data(self):
        plt.plot(self.x[self.y[:] == 0, 0].numpy(), self.x[self.y[:] == 0, 1].numpy(), 'o', label="y=0")
        plt.plot(self.x[self.y[:] == 1, 0].numpy(), self.x[self.y[:] == 1, 1].numpy(), 'ro', label="y=1")
        plt.plot(self.x[self.y[:] == 2, 0].numpy(),self.x[self.y[:] == 2, 1].numpy(), 'go',label="y=2")
        plt.legend()

<!--用于分隔主题的空格-->


<h2 id="Model">神经网络模块与训练函数</h2>


使用 <code>ModuleList()</code> 创建神经网络模块


In [ ]:
# 创建数据集对象

class Net(nn.Module):
    
    # 构造函数
    # 给定一个整数列表 Layers，我们创建神经网络的各层，其中 Layers 中的每个整数对应于该层的神经元数量
    def __init__(self, Layers):
        super(Net, self).__init__()
        self.hidden = nn.ModuleList()
        for input_size, output_size in zip(Layers, Layers[1:]):
            self.hidden.append(nn.Linear(input_size, output_size))
    
    # 预测
    # 将 X 值通过神经网络的每一层，层间使用 RELU 激活函数。最终输出不通过 RELU。
    def forward(self, x):
        L = len(self.hidden)
        for (l, linear_transform) in zip(range(L), self.hidden):
            if l < L - 1:
                x = F.relu(linear_transform(x))    
            else:
                x = linear_transform(x)
        return x

创建训练模型的函数。


In [ ]:
# 定义训练模型的函数

def train(data_set, model, criterion, train_loader, optimizer, epochs=100):
    # 用于跟踪损失和准确率的列表
    LOSS = []
    ACC = []
    # 在整个数据集上训练的次数
    for epoch in range(epochs):
        # 遍历训练数据加载器中的批次
        for x, y in train_loader:
            # 重置计算得到的梯度值，每次都必须这样做，因为如果不重置，梯度会累积
            optimizer.zero_grad()
            # 基于 X 值进行预测
            yhat = model(x)
            # 测量预测值与实际 Y 值之间的损失
            loss = criterion(yhat, y)
            # 计算每个权重和偏置的梯度值
            loss.backward()
            # 根据计算得到的梯度值更新权重和偏置
            optimizer.step()
        # 保存损失和准确率
        LOSS.append(loss.item())
        ACC.append(accuracy(model,data_set))
        
    # 绘制损失和准确率随轮次变化的图
    results ={"Loss":LOSS, "Accuracy":ACC}
    fig, ax1 = plt.subplots()
    color = 'tab:red'
    ax1.plot(LOSS,color=color)
    ax1.set_xlabel('epoch', color=color)
    ax1.set_ylabel('total loss', color=color)
    ax1.tick_params(axis = 'y', color=color)
    
    ax2 = ax1.twinx()
    color = 'tab:blue'
    ax2.set_ylabel('accuracy', color=color)  # we already handled the x-label with ax1
    ax2.plot(ACC, color=color)
    ax2.tick_params(axis='y', color=color)
    fig.tight_layout()  # otherwise the right y-label is slightly clipped
    
    plt.show()
    return results

定义用于计算准确率的函数。


In [ ]:
# 定义计算准确率的函数

def accuracy(model, data_set):
    _, yhat = torch.max(model(data_set.x), 1)
    return (yhat == data_set.y).numpy().mean()

<!--用于分隔主题的空格-->


<h2 id="Train">训练具有不同动量参数值的神经网络</h2>


使用 <code>Data</code> 创建数据集对象


In [ ]:
# 创建数据集并绘制它

data_set = Data()
data_set.plot_data()
data_set.y = data_set.y.view(-1)

用于记录不同动量参数值在每个轮次的代价和准确率值的字典。


In [ ]:
# 初始化一个包含代价和准确率的字典

Results = {"momentum 0": {"Loss": 0, "Accuracy:": 0}, "momentum 0.1": {"Loss": 0, "Accuracy:": 0}}

创建一个具有 1 个隐藏层、50 个神经元、动量为 0 的网络来对三个类别进行分类。


In [ ]:
# 训练一个具有 1 个隐藏层和 50 个神经元的模型

# 输入层大小为 2，隐藏层大小为 50，输出层大小为 3
# 我们的 X 值是 x 和 y 坐标，本问题有 3 个类别
Layers = [2, 50, 3]
# 创建一个模型
model = Net(Layers)
learning_rate = 0.10
# 创建一个使用学习率、梯度和无动量更新模型参数的优化器
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
# 为训练数据创建一个批量大小为 20 的数据加载器
train_loader = DataLoader(dataset=data_set, batch_size=20)
# 创建一个用于测量损失的准则
criterion = nn.CrossEntropyLoss()
# 使用训练函数训练模型 100 个轮次
Results["momentum 0"] = train(data_set, model, criterion, train_loader, optimizer, epochs=100)
# 打印数据集和决策边界
plot_decision_regions_3class(model, data_set)

创建一个具有 1 个隐藏层、50 个神经元、动量为 0.1 的网络来对三个类别进行分类。




In [ ]:
# 训练一个具有 1 个隐藏层和 50 个神经元的模型，动量为 0.1

# 输入层大小为 2，隐藏层大小为 50，输出层大小为 3
# 我们的 X 值是 x 和 y 坐标，本问题有 3 个类别
Layers = [2, 50, 3]
# 创建一个模型
model = Net(Layers)
learning_rate = 0.10
# 创建一个使用学习率、梯度和 0.1 动量更新模型参数的优化器
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, momentum=0.1)
# 为训练数据创建一个批量大小为 20 的数据加载器
train_loader = DataLoader(dataset=data_set, batch_size=20)
# 创建一个用于测量损失的准则
criterion = nn.CrossEntropyLoss()
# 使用训练函数训练模型 100 个轮次
Results["momentum 0.1"] = train(data_set, model, criterion, train_loader, optimizer, epochs=100)
# 打印数据集和决策边界
plot_decision_regions_3class(model, data_set)


创建一个具有 1 个隐藏层、50 个神经元、动量为 0.2 的网络来对三个类别进行分类。


In [ ]:
# 训练一个具有 1 个隐藏层和 50 个神经元的模型，动量为 0.2

# 输入层大小为 2，隐藏层大小为 50，输出层大小为 3
# 我们的 X 值是 x 和 y 坐标，本问题有 3 个类别
Layers = [2, 50, 3]
# 创建一个模型
model = Net(Layers)
learning_rate = 0.10
# 创建一个使用学习率、梯度和 0.2 动量更新模型参数的优化器
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, momentum=0.2)
# 为训练数据创建一个批量大小为 20 的数据加载器
train_loader = DataLoader(dataset=data_set, batch_size=20)
# 创建一个用于测量损失的准则
criterion = nn.CrossEntropyLoss()
# 使用训练函数训练模型 100 个轮次
Results["momentum 0.2"] = train(data_set, model, criterion, train_loader, optimizer, epochs=100)
# 打印数据集和决策边界
plot_decision_regions_3class(model, data_set)

创建一个具有 1 个隐藏层、50 个神经元、动量为 0.4 的网络来对三个类别进行分类。


In [ ]:
# 训练一个具有 1 个隐藏层和 50 个神经元的模型，动量为 0.4

# 输入层大小为 2，隐藏层大小为 50，输出层大小为 3
# 我们的 X 值是 x 和 y 坐标，本问题有 3 个类别
Layers = [2, 50, 3]
# 创建一个模型
model = Net(Layers)
learning_rate = 0.10
# 创建一个使用学习率、梯度和 0.4 动量更新模型参数的优化器
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, momentum=0.4)
# 为训练数据创建一个批量大小为 20 的数据加载器
train_loader = DataLoader(dataset=data_set, batch_size=20)
# 创建一个用于测量损失的准则
criterion = nn.CrossEntropyLoss()
# 使用训练函数训练模型 100 个轮次
Results["momentum 0.4"] = train(data_set, model, criterion, train_loader, optimizer, epochs=100)
# 打印数据集和决策边界
plot_decision_regions_3class(model, data_set)

创建一个具有 1 个隐藏层、50 个神经元、动量为 0.5 的网络来对三个类别进行分类。


In [ ]:
# 训练一个具有 1 个隐藏层和 50 个神经元的模型，动量为 0.5

# 输入层大小为 2，隐藏层大小为 50，输出层大小为 3
# 我们的 X 值是 x 和 y 坐标，本问题有 3 个类别
Layers = [2, 50, 3]
# 创建一个模型
model = Net(Layers)
learning_rate = 0.10
# 创建一个使用学习率、梯度和 0.5 动量更新模型参数的优化器
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, momentum=0.5)
# 为训练数据创建一个批量大小为 20 的数据加载器
train_loader = DataLoader(dataset=data_set, batch_size=20)
# 创建一个用于测量损失的准则
criterion = nn.CrossEntropyLoss()
# 使用训练函数训练模型 100 个轮次
Results["momentum 0.5"] = train(data_set, model, criterion, train_loader, optimizer, epochs=100)
# 打印数据集和决策边界
plot_decision_regions_3class(model, data_set)

<!--用于分隔主题的空格-->


<h2 id="Result">比较不同动量项的结果</h2>


下图比较了不同动量项的结果。总体来看，代价随着动量项的增大而成比例下降，但较大的动量项会导致更大的震荡。虽然动量项下降得更快，但似乎动量为 0.2 时达到了最小的代价值。


In [ ]:
# 绘制每个动量项的损失结果

for key, value in Results.items():
    plt.plot(value['Loss'],label=key)
    plt.legend()
    plt.xlabel('epoch')
    plt.ylabel('Total Loss or Cost')

准确率似乎与动量项成正比。


In [ ]:
# 绘制每个动量项的准确率结果

for key, value in Results.items():
    plt.plot(value['Accuracy'],label=key)
    plt.legend()
    plt.xlabel('epoch')
    plt.ylabel('Accuracy')

<!--用于分隔主题的空格-->


<h2>关于作者：</h2> 

<a href="https://www.linkedin.com/in/joseph-s-50398b136/">Joseph Santarcangelo</a> 拥有电气工程博士学位，他的研究重点是利用机器学习、信号处理和计算机视觉来确定视频如何影响人类认知。Joseph 自获得博士学位以来一直在 IBM 工作。


其他贡献者：<a href="https://www.linkedin.com/in/michelleccarey/">Michelle Carey</a>、<a href="https://www.linkedin.com/in/jiahui-mavis-zhou-a4537814a">Mavis Zhou</a>



<!--## Change Log

|  Date (YYYY-MM-DD) |  Version | Changed By  |  Change Description |
|---|---|---|---|
| 2020-09-23  | 2.0  | Sathya  |  Converted lab to Jupyterlab Current|
| 2020-09-23  | 2.0  | Srishti  |  Migrated Lab to Markdown and added to course repo in GitLab |-->



<hr>

## <h3 align="center"> © IBM Corporation. All rights reserved. <h3/>
